# MusicNet — pregled i priprema metapodataka

Ovdje učitavamo izvornu metadata tabelu, provjeravamo nedostajuće vrijednosti i pregledamo raspodjelu
snimaka i djela po kompozitorima. Zajednička logika pripreme nalazi se u `dataset.py`:

- `filter_metadata(metadata, ...)` priprema već učitanu tabelu i vraća filtriranu kopiju sa `work_id`.
- `load_metadata(...)` učitava CSV i poziva isti filter; koristi se u sveskama sa modelima.

Svaka sveska sa modelom može da se pokrene samostalno. Ovdje se ne pravi poseban CSV koji bi trebalo osvježavati.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.insert(0, "..")

import pandas as pd

from src.config import MIN_WORKS_PER_COMPOSER
from src.features.dataset import filter_metadata
from src.utils.paths import METADATA_PATH

## Učitavanje izvorne tabele

Svaki red opisuje jedan snimak / stav. Putanja do CSV fajla podešena je u `paths.py`.

In [2]:
metadata = pd.read_csv(METADATA_PATH)

print(f"Broj snimaka: {len(metadata)}")
print(f"Broj kompozitora: {metadata['composer'].nunique()}")
metadata.head()

Broj snimaka: 330
Broj kompozitora: 10


,id,composer,composition,movement,ensemble,source,transcriber,catalog_name,seconds
0,1727,Schubert,Piano Quintet in A major,2. Andante,Piano Quintet,European Archive,http://tirolmusic.blogspot.com/,OP114,447
1,1728,Schubert,Piano Quintet in A major,3. Scherzo: Presto,Piano Quintet,European Archive,http://tirolmusic.blogspot.com/,OP114,251
2,1729,Schubert,Piano Quintet in A major,4. Andantino - Allegretto,Piano Quintet,European Archive,http://tirolmusic.blogspot.com/,OP114,444
3,1730,Schubert,Piano Quintet in A major,5. Allegro giusto,Piano Quintet,European Archive,http://tirolmusic.blogspot.com/,OP114,368
4,1733,Schubert,Piano Sonata in A major,2. Andantino,Solo Piano,Museopen,Segundo G. Yogore,D959,546


In [3]:
metadata.isna().sum().to_frame("missing_values")

,missing_values
id,0
composer,0
composition,0
movement,0
ensemble,0
source,0
transcriber,0
catalog_name,0
seconds,0


In [4]:
metadata["composer"].unique()

<StringArray>
[ 'Schubert',    'Mozart',    'Dvorak',   'Cambini',     'Haydn',    'Brahms',
     'Faure',     'Ravel',      'Bach', 'Beethoven']
Length: 10, dtype: str

## Snimci i različita djela po kompozitoru

Jedno djelo može imati više stavova, pa broj snimaka nije isto što i broj djela.
Za izbor kompozitora koristimo broj **različitih djela**.

In [5]:
composer_summary = metadata.groupby("composer").agg(
    recordings=("id", "size"),
    works=("composition", "nunique"),
)
composer_summary.sort_values("works")

,recordings,works
composer,,
Haydn,3,1
Faure,4,1
Ravel,4,1
Dvorak,8,2
Cambini,9,3
Brahms,24,8
Schubert,30,9
Mozart,24,11
Bach,67,30


## Filtriranje i grupisanje

`filter_metadata` dodaje `work_id = composer + " | " + composition`, tako da svi stavovi istog djela
dobijaju istu grupu za evaluaciju. Podrazumijevani prag dolazi iz `MIN_WORKS_PER_COMPOSER` u `config.py`.
Funkcija vraća novu tabelu; izvorni `metadata` ostaje nepromijenjen.

Parametrom `exclude_composers=["Brahms"]` može se dodatno izostaviti kompozitor u pojedinom eksperimentu.

In [6]:
metadata_filtered = filter_metadata(
    metadata,
    min_works_per_composer=MIN_WORKS_PER_COMPOSER,
)

print(f"Zadržano snimaka: {len(metadata_filtered)} / {len(metadata)}")
print(f"Zadržano kompozitora: {metadata_filtered['composer'].nunique()}")
print(f"Različitih djela: {metadata_filtered['work_id'].nunique()}")
metadata_filtered.head()

Zadržano snimaka: 302 / 330
Zadržano kompozitora: 5
Različitih djela: 113


,id,composer,composition,movement,ensemble,source,transcriber,catalog_name,seconds,work_id
0,1727,Schubert,Piano Quintet in A major,2. Andante,Piano Quintet,European Archive,http://tirolmusic.blogspot.com/,OP114,447,Schubert | Piano Quintet in A major
1,1728,Schubert,Piano Quintet in A major,3. Scherzo: Presto,Piano Quintet,European Archive,http://tirolmusic.blogspot.com/,OP114,251,Schubert | Piano Quintet in A major
2,1729,Schubert,Piano Quintet in A major,4. Andantino - Allegretto,Piano Quintet,European Archive,http://tirolmusic.blogspot.com/,OP114,444,Schubert | Piano Quintet in A major
3,1730,Schubert,Piano Quintet in A major,5. Allegro giusto,Piano Quintet,European Archive,http://tirolmusic.blogspot.com/,OP114,368,Schubert | Piano Quintet in A major
4,1733,Schubert,Piano Sonata in A major,2. Andantino,Solo Piano,Museopen,Segundo G. Yogore,D959,546,Schubert | Piano Sonata in A major


In [7]:
metadata_filtered.groupby("composer").agg(
    recordings=("id", "size"),
    works=("work_id", "nunique"),
).sort_values("works")

,recordings,works
composer,,
Brahms,24,8
Schubert,30,9
Mozart,24,11
Bach,67,30
Beethoven,157,55


## Upotreba u sveskama sa modelima

```python
from src.features.dataset import load_metadata, make_dataset
from src.utils.paths import AUDIO_DIR

metadata_filtered = load_metadata()
X, y, groups = make_dataset(metadata_filtered, AUDIO_DIR)
```

Po potrebi se parametri mijenjaju u pozivu:

```python
metadata_filtered = load_metadata(
    min_works_per_composer=10,
    exclude_composers=["Brahms"],
)
```

Ista tabela se prosljeđuje funkcijama `make_spectrogram_dataset`, `make_midi_dataset` ili
`make_panns_dataset`, zavisno od eksperimenta. Parametri segmentacije ostaju u pozivu odgovarajuće funkcije.